# RTMDet-tiny 합성 통합 학습 (3팀)

**모델**: RTMDet-tiny (118 classes)
**데이터**: K-code (8302/2090) + Synthetic Combined (7200/800)
**학습**: max_epochs=15, batch=16, lr=2.5e-4, seed=100
**평가**: 33장 test set (Top-1 / Top-3 / Detection rate)

본인 이름만 ## 3 셀의 `AUTHOR`에 입력하면 본인 Drive 폴더로 자동 백업됩니다.

## 1. PyTorch 다운그레이드 (런타임 재시작 필요)

mmcv 2.1.0 호환 환경 (PyTorch 2.1.0 + CUDA 11.8 + numpy<2). 실행 후 런타임 재시작 후 다음 셀로.

In [ ]:
!pip install torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118 -q
!pip install "numpy<2" -q
print("=== 다운그레이드 완료. 런타임 재시작 후 다음 셀 실행 ===")

## 2. 다운그레이드 검증

재시작 후 가장 먼저 실행.

In [ ]:
import sys, torch, numpy
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print(f'numpy: {numpy.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 3. AUTHOR 설정 + Drive API 인증

본인 이름으로 변경 후 실행. 'pill_detection_3팀' 폴더 공유받았는지 사전 확인.

In [ ]:
# ────── 본인 이름으로 변경 ──────
AUTHOR = 'wony98'
RUN_TAG = 'v6'
# ─────────────────────────────────

import os
WORK_DIR = f'/content/rtmdet_work_{RUN_TAG}'
os.makedirs(WORK_DIR, exist_ok=True)

# Drive API 인증
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

# 'pill_detection_3팀' 폴더 ID
results = drive_service.files().list(
    q="name='pill_detection_3팀' and mimeType='application/vnd.google-apps.folder' and trashed=false",
    fields='files(id,name)'
).execute()
team_folders = results.get('files', [])
assert team_folders, "'pill_detection_3팀' 폴더 없음. 공유받았는지 확인해주세요"
TEAM_FOLDER_ID = team_folders[0]['id']

# 본인 폴더 ID
results = drive_service.files().list(
    q=f"'{TEAM_FOLDER_ID}' in parents and name='{AUTHOR}' and trashed=false",
    fields='files(id,name)'
).execute()
my_folders = results.get('files', [])
assert my_folders, f"'{AUTHOR}' 폴더 없음. Drive에서 'pill_detection_3팀/{AUTHOR}/' 폴더를 먼저 생성해주세요"
MY_FOLDER_ID = my_folders[0]['id']

print(f'AUTHOR        : {AUTHOR}')
print(f'WORK_DIR      : {WORK_DIR}')
print(f'TEAM_FOLDER_ID: {TEAM_FOLDER_ID}')
print(f'MY_FOLDER_ID  : {MY_FOLDER_ID}')

## 4. mmdet 환경 설치

mmcv 2.1.0 + mmdetection 소스 설치.

In [ ]:
!pip install -U openmim -q
!mim install mmengine -q
!mim install "mmcv>=2.0.0,<2.2.0" -q

import os
if not os.path.exists('/content/mmdetection'):
    !git clone https://github.com/open-mmlab/mmdetection.git /content/mmdetection

%cd /content/mmdetection
!pip install -v -e . -q
print('=== 설치 완료 ===')

## 5. import 검증

In [ ]:
import mmengine, mmcv, mmdet
print(f'mmengine: {mmengine.__version__}')
print(f'mmcv:     {mmcv.__version__}')
print(f'mmdet:    {mmdet.__version__}')

## 6. 데이터셋 다운로드 (gdown)

본인 Drive 공유 zip 두 개 다운로드 (약 4분, 75MB/s 측정값).
- `dataset_yolo.zip` (17.9GB) — K-code 학습 데이터
- `synthetic_combined.zip` (3.3GB) — 합성 통합본

gdown이 한도 차단(`FileURLRetrievalError: Too many users have viewed`) 뜨면 24시간 대기 또는 본인 Drive에 사본 만들어 새 file_id로 시도.

In [ ]:
import os, time
from googleapiclient.http import MediaIoBaseDownload

DOWNLOADS = {
    'dataset_yolo.zip':  '1miPs4XRtLP1noLnfSQrbQyA-95XnKIM0',  # 그대로
    'synthetic_v2.zip':  '1s0NGddiMn9IjtphZqYBWF0AaUEBOQq_g',  # ← zip 업로드 끝나면 채우기
}

for fname, fid in DOWNLOADS.items():
    out = f'/content/{fname}'
    if os.path.exists(out) and os.path.getsize(out) > 1e9:  # 1GB 이상이면 완료된 것으로 간주
        print(f'{fname}: 이미 있음 ({os.path.getsize(out)/1e9:.1f} GB), 스킵')
        continue

    print(f'\n=== {fname} 다운로드 (Drive API) ===')
    request = drive_service.files().get_media(fileId=fid)

    t0 = time.time()
    with open(out, 'wb') as fp:
        downloader = MediaIoBaseDownload(fp, request, chunksize=200*1024*1024)
        done = False
        last_pct = -1
        while not done:
            status, done = downloader.next_chunk()
            if status:
                pct = int(status.progress() * 100)
                if pct >= last_pct + 5:
                    print(f'  {pct}%')
                    last_pct = pct
    elapsed = time.time() - t0
    size_gb = os.path.getsize(out) / 1e9
    print(f'완료: {size_gb:.1f} GB / {elapsed:.0f}초 / {size_gb*1024/elapsed:.0f} MB/s')

## 7. 압축 해제 + 카운트 검증

dataset_yolo (8302/2090) + synthetic_combined (7200/800) 둘 다 한 번에 풀고 검증.

In [ ]:
import os, zipfile, time, glob

EXTRACT_PLAN = [
    ('/content/dataset_yolo.zip', '/content/dataset_yolo'),
    ('/content/synthetic_v2.zip', '/content/synthetic_v2'),  # ← 변경
]

for zp, outdir in EXTRACT_PLAN:
    if os.path.exists(outdir) and os.listdir(outdir):
        print(f'{os.path.basename(outdir)}: 이미 풀려있음, 스킵')
        continue
    print(f'\n=== {os.path.basename(zp)} 압축 해제 ===')
    t0 = time.time()
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(outdir)
    print(f'완료: {time.time()-t0:.0f}초')

# 카운트 검증
KCODE = '/content/dataset_yolo'
SYNTH = '/content/synthetic_v2'  # ← 변경
k_train = len(glob.glob(f'{KCODE}/images/train/*'))
k_val   = len(glob.glob(f'{KCODE}/images/val/*'))
s_train = len(glob.glob(f'{SYNTH}/images/train/*'))
s_val   = len(glob.glob(f'{SYNTH}/images/val/*'))

print(f'\nK-code train/val: {k_train} / {k_val}  (목표 8302/2090)')
print(f'Synth  train/val: {s_train} / {s_val}  (목표 7194/800)')  # ← 변경
assert (k_train, k_val, s_train, s_val) == (8302, 2090, 7194, 800), '카운트 불일치'  # ← 변경
print('\n카운트 OK')

In [ ]:
import os, shutil
from glob import glob
from tqdm import tqdm

BASE = '/content/synthetic_v2'

# 백슬래시 들어간 파일 모두 찾기
bad_files = glob(f'{BASE}/*\\*')  # synthetic_v2 바로 아래에 \ 포함된 것들
print(f'백슬래시 파일: {len(bad_files)}개')

moved = 0
for src in tqdm(bad_files):
    fname = os.path.basename(src)  # 예: 'synthetic_v2\\images\\train\\synth_00000.jpg'
    # 'synthetic_v2\' 접두사 제거 후 \ → / 변환
    rel = fname.replace('synthetic_v2\\', '').replace('\\', '/')
    dst = os.path.join(BASE, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.move(src, dst)
    moved += 1

print(f'이동 완료: {moved}개')

# 검증
import os
for split in ['train', 'val']:
    n_img = len(os.listdir(f'{BASE}/images/{split}'))
    n_lbl = len(os.listdir(f'{BASE}/labels/{split}'))
    print(f'{split}: images={n_img}, labels={n_lbl}')

## 8. 클래스 이름 로드 (118종)

In [ ]:
import yaml

with open('/content/dataset_yolo/pill.yaml', 'r', encoding='utf-8') as f:
    pill_yaml = yaml.safe_load(f)

CLASS_NAMES = pill_yaml['names']
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES))]

print(f'클래스 수: {len(CLASS_NAMES)}')
print(f'[0]   : {CLASS_NAMES[0]}')
print(f'[117] : {CLASS_NAMES[117]}')
assert len(CLASS_NAMES) == 118

## 9. YOLO → COCO 포맷 변환

K-code train + synthetic train 통합 → train_coco.json. 파일명 중복 회피를 위해 abs_path를 file_name으로 사용.

In [ ]:
import os, json, glob
from PIL import Image
from tqdm import tqdm

categories = [{'id': i+1, 'name': name, 'supercategory': 'pill'}
              for i, name in enumerate(CLASS_NAMES)]


def yolo_to_coco(image_dirs, label_dirs, output_path):
    images, annotations = [], []
    img_id, ann_id = 1, 1

    for img_dir, lbl_dir in zip(image_dirs, label_dirs):
        img_files = sorted(glob.glob(f'{img_dir}/*'))
        for img_path in tqdm(img_files, desc=os.path.basename(img_dir)):
            fname = os.path.basename(img_path)
            stem = os.path.splitext(fname)[0]
            lbl_path = f'{lbl_dir}/{stem}.txt'

            try:
                with Image.open(img_path) as im:
                    W, H = im.size
            except Exception as e:
                print(f'이미지 로드 실패: {img_path} - {e}')
                continue

            images.append({
                'id': img_id, 'file_name': fname,
                'width': W, 'height': H,
                'abs_path': img_path,
            })

            if os.path.exists(lbl_path):
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) != 5:
                            continue
                        cls, cx, cy, w, h = map(float, parts)
                        bw, bh = w * W, h * H
                        bx = cx * W - bw / 2
                        by = cy * H - bh / 2
                        annotations.append({
                            'id': ann_id, 'image_id': img_id,
                            'category_id': int(cls) + 1,
                            'bbox': [bx, by, bw, bh],
                            'area': bw * bh, 'iscrowd': 0,
                        })
                        ann_id += 1
            img_id += 1

    coco = {'images': images, 'annotations': annotations, 'categories': categories}
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f'\n저장: {output_path}')
    print(f'images: {len(images)}, annotations: {len(annotations)}')


# Train 통합
yolo_to_coco(
    image_dirs=['/content/dataset_yolo/images/train', '/content/synthetic_v2/images/train'],  # ← v2
    label_dirs=['/content/dataset_yolo/labels/train', '/content/synthetic_v2/labels/train'],  # ← v2
    output_path='/content/train_coco.json'
)

# Val 통합
yolo_to_coco(
    image_dirs=['/content/dataset_yolo/images/val', '/content/synthetic_v2/images/val'],  # ← v2
    label_dirs=['/content/dataset_yolo/labels/val', '/content/synthetic_v2/labels/val'],  # ← v2
    output_path='/content/val_coco.json'
)

## 10. file_name → abs_path 교체

mmdet이 절대경로로 이미지를 찾을 수 있도록 file_name을 abs_path로 교체.

In [ ]:
import json

for path in ['/content/train_coco.json', '/content/val_coco.json']:
    with open(path) as f:
        coco = json.load(f)

    # abs_path 처리는 이미 됐을 텐데 한 번 더 안전하게
    for img in coco['images']:
        if 'abs_path' in img:
            img['file_name'] = img.pop('abs_path')

    # info / licenses 필드 (pycocotools loadRes 호환)
    if 'info' not in coco:
        coco['info'] = {'description': 'Pill Detection 118 classes', 'version': '1.0', 'year': 2026}
    if 'licenses' not in coco:
        coco['licenses'] = []

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(coco, f, ensure_ascii=False)
    print(f'{path} 패치 완료')

## 11. RTMDet-tiny config 작성

In [ ]:
class_names = tuple(CLASS_NAMES)

config_text = f'''
# RTMDet-tiny custom config (Pill Detection 118 classes) — v6 (imgsz 1280)
_base_ = '/content/mmdetection/configs/rtmdet/rtmdet_tiny_8xb32-300e_coco.py'

# === 데이터 ===
data_root = ''
metainfo = {{
    'classes': {class_names!r},
}}

# === imgsz 1280 ===
image_scale = (1280, 1280)

# === 모델 ===
model = dict(
    bbox_head=dict(num_classes=118),
)

# === 학습 ===
max_epochs = 15
train_cfg = dict(
    max_epochs=max_epochs,
    val_interval=1,
)

# === Pipelines (imgsz 1280 명시 적용) ===
train_pipeline = [
    dict(type='LoadImageFromFile', backend_args=None),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='CachedMosaic', img_scale=image_scale, pad_val=114.0, max_cached_images=20, random_pop=False),
    dict(type='RandomResize', scale=(image_scale[0]*2, image_scale[1]*2), ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=image_scale),
    dict(type='YOLOXHSVRandomAug'),
    dict(type='RandomFlip', prob=0.5),
    dict(type='Pad', size=image_scale, pad_val=dict(img=(114, 114, 114))),
    dict(type='CachedMixUp', img_scale=image_scale, ratio_range=(1.0, 1.0), max_cached_images=10, random_pop=False, pad_val=(114, 114, 114), prob=0.5),
    dict(type='PackDetInputs'),
]

train_pipeline_stage2 = [
    dict(type='LoadImageFromFile', backend_args=None),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RandomResize', scale=image_scale, ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=image_scale),
    dict(type='YOLOXHSVRandomAug'),
    dict(type='RandomFlip', prob=0.5),
    dict(type='Pad', size=image_scale, pad_val=dict(img=(114, 114, 114))),
    dict(type='PackDetInputs'),
]

test_pipeline = [
    dict(type='LoadImageFromFile', backend_args=None),
    dict(type='Resize', scale=image_scale, keep_ratio=True),
    dict(type='Pad', size=image_scale, pad_val=dict(img=(114, 114, 114))),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='PackDetInputs', meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor')),
]

# === Dataloader ===
train_dataloader = dict(
    batch_size=16,
    num_workers=4,
    dataset=dict(
        data_root=data_root,
        ann_file='/content/train_coco.json',
        data_prefix=dict(img=''),
        metainfo=metainfo,
        pipeline=train_pipeline,
    ),
)
val_dataloader = dict(
    batch_size=16,
    num_workers=4,
    dataset=dict(
        data_root=data_root,
        ann_file='/content/val_coco.json',
        data_prefix=dict(img=''),
        metainfo=metainfo,
        pipeline=test_pipeline,
    ),
)
test_dataloader = val_dataloader

val_evaluator = dict(ann_file='/content/val_coco.json')
test_evaluator = val_evaluator

# === Stage 2 pipeline switch hook (RTMDet 마지막 epoch 직전 augmentation 약화) ===
custom_hooks = [
    dict(
        type='PipelineSwitchHook',
        switch_epoch=max_epochs - 5,
        switch_pipeline=train_pipeline_stage2)
]

# === Optimizer ===
base_lr = 0.00025
optim_wrapper = dict(
    optimizer=dict(lr=base_lr),
)

# === Seed ===
randomness = dict(seed=100)

# === Pretrained ===
load_from = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_tiny_8xb32-300e_coco/rtmdet_tiny_8xb32-300e_coco_20220902_112414-78e30dcc.pth'

# === 체크포인트 ===
default_hooks = dict(
    checkpoint=dict(interval=1, max_keep_ckpts=3, save_best='coco/bbox_mAP'),
)

work_dir = '{WORK_DIR}'
'''

config_path = f'{WORK_DIR}/rtmdet_tiny_pill.py'
with open(config_path, 'w', encoding='utf-8') as f:
    f.write(config_text)

print(f'config 저장: {config_path}')
print(f'  num_classes : 118')
print(f'  max_epochs  : 15')
print(f'  batch_size  : 16 (imgsz 1280 OOM 확인)')
print(f'  image_scale : (1280, 1280)')
print(f'  lr          : 0.00025')
print(f'  seed        : 100')

## 12. config + 모델/데이터셋 빌드 검증

In [ ]:
from mmengine.config import Config
from mmengine.registry import init_default_scope

cfg = Config.fromfile(f'{WORK_DIR}/rtmdet_tiny_pill.py')
init_default_scope('mmdet')

print('=== Config 파싱 ===')
print(f'max_epochs       : {cfg.train_cfg.max_epochs}')
print(f'train batch_size : {cfg.train_dataloader.batch_size}')
print(f'val_interval     : {cfg.train_cfg.val_interval}')
print(f'num_classes      : {cfg.model.bbox_head.num_classes}')
print(f'lr               : {cfg.optim_wrapper.optimizer.lr}')
print(f'seed             : {cfg.randomness.seed}')

print('\n=== 모델 빌드 시도 ===')
from mmdet.registry import MODELS
model = MODELS.build(cfg.model)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'모델 빌드 OK ({n_params:.1f}M params)')

print('\n=== Train dataset 빌드 시도 ===')
from mmdet.registry import DATASETS
train_ds = DATASETS.build(cfg.train_dataloader.dataset)
print(f'샘플 수: {len(train_ds)}')
sample = train_ds[0]
print(f'첫 샘플 img_path: {sample["data_samples"].img_path}')

del model, train_ds

## 13. AutoUploadDriveHook 등록 + 학습 실행

매 epoch 끝나면 work_dir 통째로 본인 Drive 폴더에 자동 백업. 도중 끊겨도 셀 재실행 시 마지막 epoch에서 resume.

In [ ]:
import os, zipfile, time
from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.hooks import Hook
from mmengine.registry import HOOKS


@HOOKS.register_module(force=True)
class AutoUploadDriveHook(Hook):
    """매 epoch 끝마다 work_dir 통째로 zip → Drive 본인 폴더에 직접 업로드"""
    priority = 'LOW'

    def __init__(self, work_dir, author, drive_folder_id, run_tag):
        self.work_dir = work_dir
        self.author = author
        self.drive_folder_id = drive_folder_id
        self.zip_name = f'{author}_{run_tag}_work_dir.zip'
        self.cached_file_id = None

    def _get_drive(self):
        from googleapiclient.discovery import build
        return build('drive', 'v3')

    def _find_existing(self, drive):
        results = drive.files().list(
            q=f"'{self.drive_folder_id}' in parents and name='{self.zip_name}' and trashed=false",
            fields='files(id)'
        ).execute()
        files = results.get('files', [])
        return files[0]['id'] if files else None

    def after_train_epoch(self, runner):
        from googleapiclient.http import MediaFileUpload

        zip_path = f'/content/{self.zip_name}'
        if os.path.exists(zip_path):
            os.remove(zip_path)

        runner.logger.info(f'[Upload] zip 생성 시작')
        t0 = time.time()
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
            for root, _, fs in os.walk(self.work_dir):
                for f in fs:
                    fp = os.path.join(root, f)
                    arc = os.path.relpath(fp, self.work_dir)
                    zf.write(fp, arc)
        size_mb = os.path.getsize(zip_path) / 1e6
        runner.logger.info(f'[Upload] zip 생성 완료 ({size_mb:.0f} MB, {time.time()-t0:.0f}초)')

        try:
            drive = self._get_drive()
            if self.cached_file_id is None:
                self.cached_file_id = self._find_existing(drive)

            media = MediaFileUpload(zip_path, resumable=True, chunksize=50*1024*1024)

            if self.cached_file_id:
                request = drive.files().update(
                    fileId=self.cached_file_id, media_body=media, fields='id,name')
            else:
                metadata = {'name': self.zip_name, 'parents': [self.drive_folder_id]}
                request = drive.files().create(
                    body=metadata, media_body=media, fields='id,name')

            t0 = time.time()
            response = None
            while response is None:
                status, response = request.next_chunk()
            runner.logger.info(f'[Upload] Drive 업로드 완료 ({time.time()-t0:.0f}초)')

            if not self.cached_file_id:
                self.cached_file_id = response['id']

        except Exception as e:
            runner.logger.warning(f'[Upload] 실패: {e}')


# config 로드 + custom hook 등록
cfg = Config.fromfile(f'{WORK_DIR}/rtmdet_tiny_pill.py')
cfg.custom_hooks = list(cfg.get('custom_hooks', [])) + [
    dict(type='AutoUploadDriveHook',
         work_dir=WORK_DIR,
         author=AUTHOR,
         drive_folder_id=MY_FOLDER_ID,
         run_tag=RUN_TAG)
]
# resume 처리
last_ckpt_file = os.path.join(WORK_DIR, 'last_checkpoint')
if os.path.exists(last_ckpt_file):
    with open(last_ckpt_file) as f:
        ckpt_path = f.read().strip()
    print(f'resume 모드: {os.path.basename(ckpt_path)}')
    cfg.resume = True
else:
    print(f'첫 학습 시작 (load_from: {os.path.basename(cfg.load_from)})')

runner = Runner.from_cfg(cfg)
runner.train()

print('\n=== 학습 종료 ===')

## 14. (필요 시) work_dir 상태 확인

In [ ]:
import glob, os

print(f'work_dir: {WORK_DIR}')
for f in sorted(os.listdir(WORK_DIR)):
    full = os.path.join(WORK_DIR, f)
    size = os.path.getsize(full)/1e6 if os.path.isfile(full) else 0
    print(f'  {f}  ({size:.1f} MB)' if size else f'  {f}/')

best = sorted(glob.glob(f'{WORK_DIR}/best_coco_bbox_mAP_epoch_*.pth'))
print(f'\nbest ckpt: {best[-1] if best else "없음"}')

## 15. best 체크포인트 + 모델 로드

In [ ]:
import torch, glob
from mmdet.apis import init_detector

best_paths = sorted(glob.glob(f'{WORK_DIR}/best_coco_bbox_mAP_epoch_*.pth'))
assert best_paths, f'best 체크포인트 없음: {WORK_DIR}'
BEST_CKPT = best_paths[-1]

CONFIG_PATH = f'{WORK_DIR}/rtmdet_tiny_pill.py'

print(f'best ckpt: {BEST_CKPT}')
print(f'config:    {CONFIG_PATH}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
infer_model = init_detector(CONFIG_PATH, BEST_CKPT, device=device)
print(f'\n모델 로드 완료 ({device})')

## 16. 평가 데이터 다운로드 (Drive API)

팀 Drive `pill_detection_3팀/eval/` 에서 33장 test_images + test_labels 다운로드. Drive 마운트 불가 환경 호환.

In [ ]:
from pathlib import Path
import os, time
from googleapiclient.http import MediaIoBaseDownload

LOCAL_EVAL = Path('/content/eval')
TEST_IMG_DIR = LOCAL_EVAL / 'test_images'
LABEL_DIR    = LOCAL_EVAL / 'test_labels'


def download_folder(parent_id, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)
    res = drive_service.files().list(
        q=f"'{parent_id}' in parents and trashed=false",
        fields='files(id,name,size)',
        pageSize=1000
    ).execute()
    files = res.get('files', [])
    print(f'  {len(files)}개 파일')
    t0 = time.time()
    for i, f in enumerate(files, 1):
        out = local_dir / f['name']
        if out.exists() and os.path.getsize(out) == int(f.get('size', 0)):
            continue
        request = drive_service.files().get_media(fileId=f['id'])
        with open(out, 'wb') as fp:
            downloader = MediaIoBaseDownload(fp, request, chunksize=10*1024*1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        if i % 10 == 0 or i == len(files):
            print(f'    {i}/{len(files)}')
    print(f'  완료: {time.time()-t0:.0f}초')


need_dl = not (TEST_IMG_DIR.exists() and LABEL_DIR.exists() and
               len(list(TEST_IMG_DIR.iterdir())) >= 33 and
               len(list(LABEL_DIR.glob('*.txt'))) >= 33)

if need_dl:
    res = drive_service.files().list(
        q=f"'{TEAM_FOLDER_ID}' in parents and name='eval' and trashed=false",
        fields='files(id,name)'
    ).execute()
    eval_folders = res.get('files', [])
    assert eval_folders, "'eval' 폴더 없음"
    EVAL_FOLDER_ID = eval_folders[0]['id']

    res = drive_service.files().list(
        q=f"'{EVAL_FOLDER_ID}' in parents and trashed=false",
        fields='files(id,name,mimeType)'
    ).execute()

    for item in res.get('files', []):
        if (item['mimeType'] == 'application/vnd.google-apps.folder'
                and item['name'] in ('test_images', 'test_labels')):
            print(f'\n=== {item["name"]} ===')
            download_folder(item['id'], LOCAL_EVAL / item['name'])
else:
    print('eval 데이터 이미 다운로드됨, 스킵')

assert TEST_IMG_DIR.exists()
assert LABEL_DIR.exists()

img_files = sorted(list(TEST_IMG_DIR.glob('*.png')) + list(TEST_IMG_DIR.glob('*.jpg')))
txt_files = sorted(LABEL_DIR.glob('*.txt'))

print(f'\n평가 이미지: {len(img_files)}장')
print(f'GT 라벨:     {len(txt_files)}개')

## 17. Top-3 그룹핑 함수

같은 박스에 여러 클래스 후보가 있을 때 conf 기준으로 Top-3 추출. IoU 그룹핑(greedy) + 클래스 중복 제거.

In [ ]:
import numpy as np


def iou_xyxy(b1, b2):
    """xyxy 형식 IoU."""
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter + 1e-9)


def extract_top3(bboxes, scores, labels, class_names,
                 conf_thr=0.05, iou_thr=0.5, topk=3):
    keep = scores >= conf_thr
    bboxes, scores, labels = bboxes[keep], scores[keep], labels[keep]
    if len(bboxes) == 0:
        return []

    order = np.argsort(-scores)
    bboxes, scores, labels = bboxes[order], scores[order], labels[order]

    n = len(bboxes)
    group_id = -np.ones(n, dtype=int)
    next_gid = 0
    for i in range(n):
        if group_id[i] >= 0:
            continue
        group_id[i] = next_gid
        for j in range(i + 1, n):
            if group_id[j] >= 0:
                continue
            if iou_xyxy(bboxes[i], bboxes[j]) > iou_thr:
                group_id[j] = next_gid
        next_gid += 1

    results = []
    for gid in range(next_gid):
        mask = group_id == gid
        g_bboxes, g_scores, g_labels = bboxes[mask], scores[mask], labels[mask]

        seen_cls = {}
        for b, s, l in zip(g_bboxes, g_scores, g_labels):
            l = int(l)
            if l not in seen_cls or s > seen_cls[l][1]:
                seen_cls[l] = (b, float(s), l)

        cls_sorted = sorted(seen_cls.values(), key=lambda x: -x[1])[:topk]
        rep_bbox = cls_sorted[0][0].tolist()

        results.append({
            'bbox': rep_bbox,
            'top3_class_name': [class_names[l] for _, _, l in cls_sorted],
            'top3_confidence': [s for _, s, _ in cls_sorted],
        })
    return results


# sanity
_b = np.array([[10,10,50,50], [12,12,48,48], [200,200,250,250]], dtype=float)
_s = np.array([0.9, 0.7, 0.6])
_l = np.array([0, 1, 2])
_names = [f'class_{i}' for i in range(118)]
_out = extract_top3(_b, _s, _l, _names)
print(f'입력 박스 3 → 그룹 {len(_out)}개 (앞 두 박스 IoU 그룹핑되어 1그룹, 세 번째 별도 1그룹)')

## 18. 33장 추론

In [ ]:
import time
from mmdet.apis import inference_detector
from tqdm.auto import tqdm

predictions_per_image = {}
times_ms = []

for img_path in tqdm(img_files, desc='33장 추론'):
    t0 = time.time()
    result = inference_detector(infer_model, str(img_path))
    times_ms.append((time.time() - t0) * 1000)

    inst = result.pred_instances
    bboxes = inst.bboxes.cpu().numpy()
    scores = inst.scores.cpu().numpy()
    labels = inst.labels.cpu().numpy()

    preds = extract_top3(bboxes, scores, labels, CLASS_NAMES,
                          conf_thr=0.05, iou_thr=0.5, topk=3)
    predictions_per_image[img_path.name] = preds

total_groups = sum(len(v) for v in predictions_per_image.values())
ms_per_image = float(np.mean(times_ms))

print(f'\n추론 완료: {len(predictions_per_image)}장')
print(f'총 그룹 수: {total_groups}')
print(f'평균 추론 시간: {ms_per_image:.1f} ms/img')

first_name = list(predictions_per_image.keys())[0]
print(f'\n[{first_name}] 그룹 {len(predictions_per_image[first_name])}개')
for i, g in enumerate(predictions_per_image[first_name]):
    print(f"  [{i}] bbox={[round(x,1) for x in g['bbox']]}")
    for cn, cf in zip(g['top3_class_name'], g['top3_confidence']):
        print(f'      {cn} ({cf:.3f})')

## 19. 결과 JSON 저장 + Drive 업로드

v4 형식으로 저장 (모델 비교 노트북 호환). Drive 본인 폴더에 함께 백업.

In [ ]:
import json
import unicodedata
from datetime import datetime


def nfc(s):
    return unicodedata.normalize('NFC', s) if isinstance(s, str) else s


best_epoch = int(Path(BEST_CKPT).stem.split('_')[-1])
n_params_M = round(sum(p.numel() for p in infer_model.parameters()) / 1e6, 1)

results_arr = []
for img_path in img_files:
    stem = nfc(img_path.stem)
    fname = img_path.name

    gt_bboxes = []
    txt_path = LABEL_DIR / f'{stem}.txt'
    if txt_path.exists():
        from PIL import Image
        img_w, img_h = Image.open(img_path).size
        with open(txt_path, encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                cx, cy, w, h = map(float, parts[1:])
                gt_bboxes.append({
                    'bbox': [(cx-w/2)*img_w, (cy-h/2)*img_h,
                             (cx+w/2)*img_w, (cy+h/2)*img_h],
                    'dl_name': nfc(CLASS_NAMES[cls]),
                })

    results_arr.append({
        'filename': fname,
        'gt_bboxes': gt_bboxes,
        'predictions': predictions_per_image.get(fname, []),
    })

output_json = {
    'exp_name':         f'rtmdet_tiny_synth_e{best_epoch}_seed100',
    'author':           AUTHOR,
    'model_framework':  'mmdet rtmdet_tiny',
    'model_params_M':   n_params_M,
    'batch_size_used':  16,
    'seed':             100,
    'conf_threshold':   0.05,
    'nms_iou':          0.5,
    'best_epoch':       best_epoch,
    'ms_per_image':     round(ms_per_image, 1),
    'eval_version':     'v4',
    'num_images':       len(results_arr),
    'num_gt_bboxes':    sum(len(r['gt_bboxes']) for r in results_arr),
    'created_at':       datetime.now().isoformat(timespec='seconds'),
    'results':          results_arr,
}

out_path = Path(WORK_DIR) / f'rtmdet_tiny_synth_e{best_epoch}_seed100_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output_json, f, ensure_ascii=False, indent=2)

print(f'저장: {out_path}')
print(f"shape: num_images={output_json['num_images']}, num_gt_bboxes={output_json['num_gt_bboxes']}")

# Drive 업로드
from googleapiclient.http import MediaFileUpload

media = MediaFileUpload(str(out_path), mimetype='application/json', resumable=False)
file_meta = {'name': out_path.name, 'parents': [MY_FOLDER_ID]}

existing = drive_service.files().list(
    q=f"'{MY_FOLDER_ID}' in parents and name='{out_path.name}' and trashed=false",
    fields='files(id)').execute().get('files', [])
if existing:
    drive_service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    print(f'Drive 덮어쓰기: {out_path.name}')
else:
    drive_service.files().create(body=file_meta, media_body=media).execute()
    print(f'Drive 업로드: {out_path.name}')

## 20. 채점 + 결과 출력

GT bbox별 IoU≥0.5 매칭 후 Top-1 / Top-3 / Detection rate 계산.

In [ ]:
def evaluate_v4_json(output_json, iou_thr=0.5):
    n_top1 = n_top3 = n_match = 0
    img_top1_full = img_top3_full = 0
    n_false_pos = 0
    total_gt = sum(len(r['gt_bboxes']) for r in output_json['results'])
    n_imgs = len(output_json['results'])

    for r in output_json['results']:
        gts = r['gt_bboxes']
        preds = r['predictions']
        used = set()
        img_all_top1 = bool(gts)
        img_all_top3 = bool(gts)

        for gt in gts:
            best_iou, best_idx = 0.0, -1
            for j, p in enumerate(preds):
                if j in used:
                    continue
                i = iou_xyxy(gt['bbox'], p['bbox'])
                if i > best_iou:
                    best_iou, best_idx = i, j

            if best_iou >= iou_thr and best_idx >= 0:
                n_match += 1
                used.add(best_idx)
                top3 = [nfc(n) for n in preds[best_idx]['top3_class_name']]
                gt_name = nfc(gt['dl_name'])
                if top3 and top3[0] == gt_name:
                    n_top1 += 1
                else:
                    img_all_top1 = False
                if gt_name in top3:
                    n_top3 += 1
                else:
                    img_all_top3 = False
            else:
                img_all_top1 = False
                img_all_top3 = False

        if img_all_top1:
            img_top1_full += 1
        if img_all_top3:
            img_top3_full += 1
        n_false_pos += len(preds) - len(used)

    return {
        'detection_rate': n_match / total_gt,
        'top1_accuracy':  n_top1  / total_gt,
        'top3_accuracy':  n_top3  / total_gt,
        'image_top1':     img_top1_full / n_imgs,
        'image_top3':     img_top3_full / n_imgs,
        'n_top1': n_top1, 'n_top3': n_top3, 'n_match': n_match,
        'n_false_pos': n_false_pos, 'total_gt': total_gt, 'n_imgs': n_imgs,
    }


metrics = evaluate_v4_json(output_json)

print('=' * 50)
print(f'  RTMDet-tiny synth_e{best_epoch} seed=100 채점 결과')
print('=' * 50)
print(f"Detection (IoU>=0.5): {metrics['n_match']}/{metrics['total_gt']} ({metrics['detection_rate']:.1%})")
print(f"Top-1 정답:           {metrics['n_top1']}/{metrics['total_gt']} ({metrics['top1_accuracy']:.1%})")
print(f"Top-3 정답:           {metrics['n_top3']}/{metrics['total_gt']} ({metrics['top3_accuracy']:.1%})")
print(f"이미지 단위 Top-1:    {int(metrics['image_top1']*metrics['n_imgs'])}/{metrics['n_imgs']} ({metrics['image_top1']:.1%})")
print(f"이미지 단위 Top-3:    {int(metrics['image_top3']*metrics['n_imgs'])}/{metrics['n_imgs']} ({metrics['image_top3']:.1%})")
print(f"오탐 그룹:            {metrics['n_false_pos']}")
print(f"평균 추론 시간:       {output_json['ms_per_image']:.1f} ms/img")

In [ ]:
# === 학습 + 평가 완료 후 런타임 자동 종료 ===
import time

print('=' * 50)
print('  모든 작업 완료. 런타임 종료 대기 중...')
print('=' * 50)

# 마지막 출력/Drive 업로드 마무리 여유
time.sleep(60)

print('런타임 종료')
from google.colab import runtime
runtime.unassign()

## 분석 B — 1순위 예측 빈도 분포 비교

baseline 핀 클래스 (Test 등장 0회인데 자주 1순위 예측):
- 엘스테인캡슐(에르도스테인) 23회 / 이가탄에프캡슐 150mg/PTP 14회 / 알바스테인캡슐(에르도스테인) 13회
- 람노스캡슐500mg/병 11회 / 메가파워정 90mg/병 10회 / 카발린캡슐 25mg 9회

검증:
- v2에서 위 6개 빈도 변화 (감소면 처방 효과)
- v2에 새 핀 클래스 (Test 등장 0회 + v2 빈도 ≥ 임계) 등장 여부
- baseline FP 259 → v2 FP 100 감소가 진짜 정확도 개선인지 단순 conf 낮아진 결과인지

대상: 매칭된 예측 + 오탐(FP) 모든 1순위 예측. results JSON의 모든 predictions 순회.